# 116 — Grand Ensemble v10

New best anchor: nb111 (enhanced_delta_3tier) OOF RAE = 0.2480.
Previous best was nb109 (delta_ensemble_blend) at 0.2748.

Strategy: ElasticNet stacking with delta family anchored by nb111,
plus top complementary non-delta models.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNetCV, RidgeCV
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko
from pxr.paths import DATA_PROCESSED, SUBMISSIONS
SEED = 42; N_FOLDS = 5
print("imports OK")

imports OK


In [2]:
from scipy import stats

def full_metrics(y_true, y_pred, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr), Spearman=float(sp))
    if label:
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} r={pr:.4f} rho={sp:.4f}")
    return m

In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)

# Known leaky OOF files (use train-only features like emax/pec50_se)
LEAKY = {
    "oof_aux_features.npy",
    "oof_creative_mega_ensemble.npy",
    "oof_grand_v6.npy",
    "oof_grand_v7.npy",
    "oof_grand_v5.npy",
}

# Load all legitimate OOF arrays
all_oof = {}
for f in sorted(DATA_PROCESSED.glob("oof_*.npy")):
    if f.name in LEAKY: continue
    try:
        arr = np.load(f)
        if len(arr) != len(y_tr): continue
        mask = np.isfinite(arr)
        if mask.sum() < 3800: continue
        r = rae(y_tr[mask], arr[mask])
        if r > 0.65: continue
        all_oof[f.stem.replace("oof_","")] = arr
    except: pass

all_te = {}
for name in all_oof:
    te_path = DATA_PROCESSED / f"te_oof_{name}.npy"
    if te_path.exists():
        arr = np.load(te_path)
        if len(arr) == len(te): all_te[name] = arr

print(f"Loaded {len(all_oof)} legitimate OOF arrays ({len(all_te)} with test preds)")

# Sort by RAE
names_by_rae = sorted(all_oof.keys(), key=lambda n: rae(y_tr, all_oof[n]))
print("\nTop-15 individual OOF RAEs:")
for n in names_by_rae[:15]:
    r = rae(y_tr, all_oof[n])
    te_flag = 'TE' if n in all_te else '--'
    print(f"  {r:.4f}  [{te_flag}]  {n}")

Loaded 118 legitimate OOF arrays (39 with test preds)

Top-15 individual OOF RAEs:
  0.2473  [TE]  blend_optimizer
  0.2480  [TE]  enhanced_delta_3tier
  0.2748  [TE]  grand_v9
  0.2748  [TE]  delta_ensemble_blend
  0.2772  [TE]  delta_similarity_tiers
  0.2843  [TE]  grand_v8
  0.2888  [TE]  delta_5tiers
  0.3266  [TE]  delta_loso
  0.3266  [TE]  multi_template_delta
  0.3268  [TE]  delta_uncertainty
  0.3269  [TE]  reverse_delta_ml
  0.3689  [--]  nb127_exhaustive_blend
  0.3693  [--]  nb125_2way
  0.3706  [--]  nb119_optuna_ensemble
  0.3714  [--]  nb112_grand_v3


In [4]:
# --- Strategy 1: Use nb111 as anchor, blend with top-5 ---
print("\n=== Strategy 1: nb111-anchored blend ===", flush=True)

# Verify nb111 is the best
best_name = names_by_rae[0]
print(f"Best single model: {best_name} (RAE={rae(y_tr, all_oof[best_name]):.4f})")

# Select top models with test predictions
top_with_te = [n for n in names_by_rae if n in all_te][:20]
print(f"Top models with test preds: {top_with_te[:10]}")

# Simple rank-weighted blend of top models
for N in [1, 2, 3, 5, 7, 10]:
    top_n = top_with_te[:N]
    raes_n = np.array([rae(y_tr, all_oof[n]) for n in top_n])
    weights = 1.0 / np.maximum(raes_n, 1e-6); weights /= weights.sum()
    blended = sum(w * all_oof[n] for w, n in zip(weights, top_n))
    mask = np.isfinite(blended)
    r = rae(y_tr[mask], blended[mask])
    print(f"  top-{N:2d}  RAE={r:.4f}  weights={weights.round(3).tolist()}")


=== Strategy 1: nb111-anchored blend ===


Best single model: blend_optimizer (RAE=0.2473)
Top models with test preds: ['blend_optimizer', 'enhanced_delta_3tier', 'grand_v9', 'delta_ensemble_blend', 'delta_similarity_tiers', 'grand_v8', 'delta_5tiers', 'delta_loso', 'multi_template_delta', 'delta_uncertainty']
  top- 1  RAE=0.2473  weights=[1.0]
  top- 2  RAE=0.2474  weights=[0.501, 0.499]
  top- 3  RAE=0.2504  weights=[0.345, 0.344, 0.311]
  top- 5  RAE=0.2565  weights=[0.213, 0.213, 0.192, 0.192, 0.19]
  top- 7  RAE=0.2591  weights=[0.156, 0.155, 0.14, 0.14, 0.139, 0.136, 0.133]
  top-10  RAE=0.2686  weights=[0.115, 0.115, 0.104, 0.104, 0.103, 0.1, 0.099, 0.087, 0.087, 0.087]


In [5]:
# --- Strategy 2: Nested-CV ElasticNet over top-15 models ---
print("\n=== Strategy 2: Nested-CV ElasticNet (top-15 with test preds) ===", flush=True)

top15 = top_with_te[:15]
OOF_stack = np.column_stack([all_oof[n] for n in top15])
TE_stack  = np.column_stack([all_te[n]  for n in top15])

oof_enet = np.full(len(y_tr), np.nan)
for k, (tr_idx, va_idx) in enumerate(splits):
    meta_tr_idx = [i for fold,(ti,_) in enumerate(splits) for i in ti if fold!=k]
    meta = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=10000,
                        random_state=SEED, n_jobs=2)
    meta.fit(OOF_stack[meta_tr_idx], y_tr[meta_tr_idx])
    oof_enet[va_idx] = meta.predict(OOF_stack[va_idx])
    fold_rae = rae(y_tr[va_idx], oof_enet[va_idx])
    print(f"  fold {k+1}  RAE={fold_rae:.4f}", flush=True)

m_enet = full_metrics(y_tr, oof_enet, "enet_v10")
print(f"\nElasticNet v10 OOF RAE: {m_enet['RAE']:.4f}")


=== Strategy 2: Nested-CV ElasticNet (top-15 with test preds) ===


  fold 1  RAE=0.2373


  fold 2  RAE=0.2504


  fold 3  RAE=0.2796


  fold 4  RAE=0.2695


  fold 5  RAE=0.2699


  [enet_v10] RAE=0.2593 MAE=0.2359 R2=0.8590 r=0.9269 rho=0.9045

ElasticNet v10 OOF RAE: 0.2593


In [6]:
# --- Strategy 3: Best 2-model blend (nb111 vs each other) ---
print("\n=== Strategy 3: Best 2-model blend with nb111 ===", flush=True)

oof111 = all_oof.get("enhanced_delta_3tier")
if oof111 is None:
    print("nb111 OOF not found")
    best2_rae = m_enet["RAE"]; best2_oof = oof_enet
else:
    r111 = rae(y_tr, oof111)
    print(f"nb111 RAE: {r111:.4f}")
    best2_results = []
    for other_name in names_by_rae[:30]:
        if other_name == "enhanced_delta_3tier" or other_name not in all_te: continue
        oof_other = all_oof[other_name]
        best_r, best_a = r111, 1.0
        for alpha in np.linspace(0, 1, 21):
            blended = alpha * oof111 + (1-alpha) * oof_other
            mask = np.isfinite(blended)
            r = rae(y_tr[mask], blended[mask])
            if r < best_r: best_r, best_a = r, alpha
        best2_results.append((best_r, best_a, other_name))
    best2_results.sort()
    print("Top-10 2-model blends with nb111:")
    for r, a, n in best2_results[:10]:
        print(f"  RAE={r:.4f}  alpha={a:.2f}  nb111 + {n}")
    best2_rae, best2_alpha, best2_other = best2_results[0]


=== Strategy 3: Best 2-model blend with nb111 ===


nb111 RAE: 0.2480
Top-10 2-model blends with nb111:
  RAE=0.2473  alpha=0.00  nb111 + blend_optimizer
  RAE=0.2473  alpha=0.95  nb111 + delta_ensemble_blend
  RAE=0.2473  alpha=0.95  nb111 + grand_v9
  RAE=0.2480  alpha=1.00  nb111 + delta_5tiers
  RAE=0.2480  alpha=1.00  nb111 + delta_loso
  RAE=0.2480  alpha=1.00  nb111 + delta_similarity_tiers
  RAE=0.2480  alpha=1.00  nb111 + delta_uncertainty
  RAE=0.2480  alpha=1.00  nb111 + grand_v8
  RAE=0.2480  alpha=1.00  nb111 + multi_template_delta
  RAE=0.2480  alpha=1.00  nb111 + reverse_delta_ml


In [7]:
# --- Pick best approach and save ---
print("\n=== Comparison ===")
approaches = [
    ("nb111_single",   rae(y_tr, all_oof[best_name]) if best_name in all_oof else float("inf")),
    ("enet_v10",        m_enet["RAE"]),
]
if oof111 is not None and best2_results:
    approaches.append(("2model_w_111", best2_rae))
approaches.sort(key=lambda x: x[1])
for name, r in approaches:
    print(f"  {r:.4f}  {name}")

best_approach = approaches[0][0]
best_rae_v10 = approaches[0][1]
print(f"\nBest approach: {best_approach}  RAE={best_rae_v10:.4f}")
print(f"Previous best (grand_v9): 0.2748")
print(f"Improvement: {0.2748 - best_rae_v10:+.4f}")

# Build final OOF and test preds
if "enet" in best_approach:
    OOF_final = oof_enet
    meta_final = ElasticNetCV(l1_ratio=[0.1,0.5,0.9,1.0], cv=5, max_iter=10000, random_state=SEED)
    meta_final.fit(OOF_stack, y_tr)
    TE_final = np.clip(meta_final.predict(TE_stack), y_tr.min()-0.5, y_tr.max()+0.5)
    coef_df = pd.DataFrame({"model": top15, "weight": meta_final.coef_}).sort_values("weight", ascending=False)
    print("\nTop weights:")
    print(coef_df[coef_df.weight.abs()>0.01].to_string(index=False))
elif "2model" in best_approach and oof111 is not None:
    te111 = all_te.get("enhanced_delta_3tier")
    te_other = all_te.get(best2_other)
    OOF_final = best2_alpha * oof111 + (1-best2_alpha) * all_oof[best2_other]
    TE_final = best2_alpha * te111 + (1-best2_alpha) * te_other
    TE_final = np.clip(TE_final, y_tr.min()-0.5, y_tr.max()+0.5)
else:
    OOF_final = all_oof[best_name]
    TE_final = np.clip(all_te[best_name], y_tr.min()-0.5, y_tr.max()+0.5)

np.save(DATA_PROCESSED/"oof_grand_v10.npy", OOF_final)
np.save(DATA_PROCESSED/"te_oof_grand_v10.npy", TE_final)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": TE_final})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"116_grand_ensemble_v10.csv"; sub.to_csv(p, index=False)
print(f"\nSaved {p}")
print(f"Test: min={TE_final.min():.2f} med={np.median(TE_final):.2f} max={TE_final.max():.2f}")
print(f"\n*** Grand v10 OOF RAE = {best_rae_v10:.4f} ***")


=== Comparison ===
  0.2473  nb111_single
  0.2473  2model_w_111
  0.2593  enet_v10

Best approach: nb111_single  RAE=0.2473
Previous best (grand_v9): 0.2748
Improvement: +0.0275

Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\116_grand_ensemble_v10.csv
Test: min=3.29 med=4.90 max=6.54

*** Grand v10 OOF RAE = 0.2473 ***
